In [4]:
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
from tree_sitter import Language, Parser, Tree, Query, QueryCursor

class UniversalASTReader:
    """
    Core Module (Reader Agent): Updated to the latest Tree-sitter API.
    """
    def __init__(self):
        # 1. Load language "dictionary"
        self.languages = {
            "python": Language(tspython.language()),
            "java": Language(tsjava.language())
        }

        # 2. DO NOT initialize an empty Parser here anymore
        # (Remove the old self.parser = Parser() line)

    def parse(self, source_code: bytes, lang: str) -> Tree:
        """Parses the source code based on the specified language."""
        lang_key = lang.lower()
        if lang_key not in self.languages:
            raise ValueError(f"Language '{lang}' is not yet supported in Mygrate!")

        # BUG FIX HERE: Pass Language directly to Parser during initialization
        parser = Parser(self.languages[lang_key])

        return parser.parse(source_code)

    def query(self, tree: Tree, lang: str, query_string: str) -> list:
        """Executes Tree-sitter Query on the AST tree."""
        lang_key = lang.lower()
        lang_obj = self.languages[lang_key]

        query_obj = Query(lang_obj, query_string)
        cursor = QueryCursor(query_obj)
        captures_dict = cursor.captures(tree.root_node)

        # Convert new dict format {'tag': [node1, node2]} to old list format [(node, 'tag')]
        results = []
        for tag, nodes in captures_dict.items():
            for node in nodes:
                results.append((node, tag))

        # Sort by node position in the file (matching old list behavior)
        results.sort(key=lambda x: x[0].start_byte)
        return results

    def visualize(self, node, source_code: bytes, level=0):
        """Recursive function to print the Tree-sitter tree in directory format."""
        indent = "    " * level

        # If it's a leaf node (no children), print its actual text
        if len(node.children) == 0:
            text = source_code[node.start_byte:node.end_byte].decode('utf-8')
            print(f"{indent}├── {node.type} (text: '{text}')")
        else:
            # If it has children, only print the node type
            print(f"{indent}├── {node.type}")

        # Recursively call for child nodes
        for child in node.children:
            self.visualize(child, source_code, level + 1)

# ==========================================
# REAL-WORLD TEST SECTION
# ==========================================
if __name__ == "__main__":
    reader = UniversalASTReader()

    # --- TEST PYTHON ---
    py_code = b"""
def calculate_score(points: int) -> int:
    return points + 10
    """
    py_tree = reader.parse(py_code, "python")
    py_query = "(function_definition name: (identifier) @python_func)"
    py_results = reader.query(py_tree, "python", py_query)

    print("=== PYTHON PARSING RESULTS ===")
    for node, tag in py_results:
        text = py_code[node.start_byte:node.end_byte].decode('utf-8')
        print(f"🎯 Captured [{tag}]: '{text}'")

    # --- TEST JAVA ---
    java_code = b"""
public class MathUtils {
    @Test
    public void testAddition() {
        assertEquals(2, 1 + 1);
    }
}
    """
    java_tree = reader.parse(java_code, "java")
    java_query = "(method_declaration name: (identifier) @java_method)"
    java_results = reader.query(java_tree, "java", java_query)

    print("\n=== JAVA PARSING RESULTS ===")
    for node, tag in java_results:
        text = java_code[node.start_byte:node.end_byte].decode('utf-8')
        print(f"🎯 Captured [{tag}]: '{text}'")

# Visualizing
print("=== TREE-SITTER AST VISUALIZATION (PYTHON) ===")
reader.visualize(py_tree.root_node, py_code)
print("\n" + "="*40 + "\n")

=== PYTHON PARSING RESULTS ===
🎯 Captured [python_func]: 'calculate_score'

=== JAVA PARSING RESULTS ===
🎯 Captured [java_method]: 'testAddition'
=== TREE-SITTER AST VISUALIZATION (PYTHON) ===
├── module
    ├── function_definition
        ├── def (text: 'def')
        ├── identifier (text: 'calculate_score')
        ├── parameters
            ├── ( (text: '(')
            ├── typed_parameter
                ├── identifier (text: 'points')
                ├── : (text: ':')
                ├── type
                    ├── identifier (text: 'int')
            ├── ) (text: ')')
        ├── -> (text: '->')
        ├── type
            ├── identifier (text: 'int')
        ├── : (text: ':')
        ├── block
            ├── return_statement
                ├── return (text: 'return')
                ├── binary_operator
                    ├── identifier (text: 'points')
                    ├── + (text: '+')
                    ├── integer (text: '10')


